In [1]:
# 1) Locate offline assets, install the pinned Transformers build, and configure training.
from pathlib import Path
import gc, json, math, os, shutil, subprocess, sys, time
from collections import defaultdict
from dataclasses import dataclass
from importlib.metadata import PackageNotFoundError, version as dist_version

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['WANDB_DISABLED'] = 'true'

ASSET_ROOT_OVERRIDE = None
DATASET_ROOT_OVERRIDE = None

def find_asset_root():
    if ASSET_ROOT_OVERRIDE:
        root = Path(ASSET_ROOT_OVERRIDE)
        if not (root / 'manifest.json').is_file():
            raise FileNotFoundError(root / 'manifest.json')
        return root
    hits = [
        path.parent for path in Path('/kaggle/input').rglob('manifest.json')
        if (path.parent / 'models/rtdetr_r50vd_coco_o365/config.json').is_file()
    ]
    if len(hits) != 1:
        print('Asset candidates:', *hits, sep='\n  ')
        raise RuntimeError('Attach exactly one successful uav_onl notebook output.')
    return hits[0]

ASSET_ROOT = find_asset_root()
MANIFEST = json.loads((ASSET_ROOT / 'manifest.json').read_text(encoding='utf-8'))
EXPECTED_MODEL_ID = 'PekingU/rtdetr_r50vd_coco_o365'
if MANIFEST.get('model_id') != EXPECTED_MODEL_ID:
    raise RuntimeError(f'Expected {EXPECTED_MODEL_ID}, got {MANIFEST.get("model_id")}')
MODEL_DIR = ASSET_ROOT / 'models' / MANIFEST['model_folder']
WHEEL_DIR = ASSET_ROOT / 'wheels'
for required_path in (MODEL_DIR / 'config.json', MODEL_DIR / 'model.safetensors'):
    if not required_path.is_file():
        raise FileNotFoundError(required_path)

required_transformers = MANIFEST['transformers_version']
try:
    installed_transformers = dist_version('transformers')
except PackageNotFoundError:
    installed_transformers = None
missing_runtime_packages = []
for package_name in ('pycocotools', 'scipy'):
    try:
        dist_version(package_name)
    except PackageNotFoundError:
        missing_runtime_packages.append(package_name)
if 'transformers' in sys.modules and installed_transformers != required_transformers:
    raise RuntimeError('Restart Session, then Run All: a different Transformers version is already imported.')
if installed_transformers != required_transformers or missing_runtime_packages:
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '--no-index', '--upgrade',
        f'--find-links={WHEEL_DIR}', f'transformers=={required_transformers}',
        'pycocotools', 'scipy'
    ], check=True)

import numpy as np
from PIL import Image, ImageOps
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.ops import batched_nms
from tqdm.auto import tqdm
import transformers
from transformers import RTDetrForObjectDetection

SEED = 42
IMAGE_SIZE = 640
PAD_VALUE = 114
NUM_CLASSES = 10
CLASS_NAMES = [
    'pedestrian', 'people', 'bicycle', 'car', 'van', 'truck',
    'tricycle', 'awning-tricycle', 'bus', 'motor'
]
EPOCHS = 20
BATCH_SIZE = 48
GRAD_ACCUM_STEPS = 1
NUM_WORKERS = 4
BACKBONE_FROZEN_EPOCHS = 1
EVAL_EVERY = 1
LR_HEAD = 1e-4
LR_CORE = 5e-5
LR_BACKBONE = 1e-5
WEIGHT_DECAY = 1e-4
MAX_GRAD_NORM = 0.1
HORIZONTAL_FLIP_PROBABILITY = 0.5
TRAIN_TILE_SIZE = 640
TRAIN_SLICE_PROBABILITY = 0.7
TRAIN_SLICE_OBJECT_GUIDED_PROBABILITY = 0.8
TRAIN_SLICE_MIN_VISIBILITY = 0.5
TRAIN_SLICE_MIN_BOX_SIZE = 2.0
TRAIN_EMPTY_SLICE_PROBABILITY = 0.1
GOIS_TILE_SIZE = 640
GOIS_OVERLAP = 0.2
GOIS_NMS_IOU = 0.5
GOIS_SCORE_THRESHOLD = 0.001
GOIS_MAX_DETECTIONS = 500
GOIS_TILE_BATCH_SIZE = 2
RESUME_CHECKPOINT = None

WORK_DIR = Path('/kaggle/working/rtdetr_r50vd_o365_visdrone')
COCO_DIR = WORK_DIR / 'coco'
CHECKPOINT_DIR = WORK_DIR / 'checkpoints'
PRED_DIR = WORK_DIR / 'predictions'
for directory in (COCO_DIR, CHECKPOINT_DIR, PRED_DIR):
    directory.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BF16 = DEVICE.type == 'cuda' and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if BF16 else (torch.float16 if DEVICE.type == 'cuda' else None)
print('assets:', ASSET_ROOT)
print('model:', MODEL_DIR)
print('torch:', torch.__version__, 'transformers:', transformers.__version__, 'device:', DEVICE)

Looking in links: /kaggle/input/notebooks/nhtlnguyn1106/visdrone-prepare/uav_detr_offline_assets/wheels
Processing /kaggle/input/notebooks/nhtlnguyn1106/visdrone-prepare/uav_detr_offline_assets/wheels/transformers-5.15.0-py3-none-any.whl
Processing /kaggle/input/notebooks/nhtlnguyn1106/visdrone-prepare/uav_detr_offline_assets/wheels/scipy-1.18.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
Processing /kaggle/input/notebooks/nhtlnguyn1106/visdrone-prepare/uav_detr_offline_assets/wheels/safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (from transformers==5.15.0)
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
  Attempting uninstall: transformers
    Found existing installation

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires scipy<1.17,>=1.8, but you have scipy 1.18.1 which is incompatible.


assets: /kaggle/input/notebooks/nhtlnguyn1106/visdrone-prepare/uav_detr_offline_assets
model: /kaggle/input/notebooks/nhtlnguyn1106/visdrone-prepare/uav_detr_offline_assets/models/rtdetr_r50vd_coco_o365
torch: 2.10.0+cu128 transformers: 5.15.0 device: cuda


In [2]:
# 2) Discover raw VisDrone splits and convert annotations to internal COCO JSON.
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp'}

def discover_raw_splits():
    base = Path(DATASET_ROOT_OVERRIDE) if DATASET_ROOT_OVERRIDE else Path('/kaggle/input')
    candidates = []
    for annotations in base.rglob('annotations'):
        parent = annotations.parent
        images = parent / 'images'
        if images.is_dir() and any(path.suffix.lower() in IMAGE_EXTENSIONS for path in images.iterdir()):
            candidates.append(parent)
    candidates = sorted(set(candidates), key=lambda path: (len(path.parts), str(path)))
    def choose(kind):
        for path in candidates:
            name = str(path).lower()
            if kind == 'train' and 'train' in name and 'val' not in name:
                return path
            if kind == 'val' and 'val' in name:
                return path
            if kind == 'test' and ('test-dev' in name or 'test_dev' in name):
                return path
        return None
    result = {kind: choose(kind) for kind in ('train', 'val', 'test')}
    if any(result[kind] is None for kind in ('train', 'val', 'test')):
        print('Detected split candidates:', *candidates, sep='\n  ')
        raise RuntimeError('VisDrone train, val, and test-dev splits are all required.')
    return result

RAW_SPLITS = discover_raw_splits()
print({key: str(value) for key, value in RAW_SPLITS.items()})

def convert_visdrone_to_coco(root, output_path):
    images, annotations = [], []
    annotation_id = 1
    paths = sorted(path for path in (root / 'images').iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)
    for image_id, image_path in enumerate(tqdm(paths, desc=f'COCO {root.name}'), 1):
        with Image.open(image_path) as image:
            width, height = image.size
        images.append({'id': image_id, 'file_name': image_path.name, 'width': width, 'height': height})
        txt = root / 'annotations' / f'{image_path.stem}.txt'
        if not txt.is_file():
            continue
        for line in txt.read_text(encoding='utf-8-sig').splitlines():
            parts = [value.strip() for value in line.split(',')]
            if len(parts) < 6:
                continue
            try:
                x, y, box_width, box_height = map(float, parts[:4])
                valid, category = int(float(parts[4])), int(float(parts[5]))
            except ValueError:
                continue
            if valid <= 0 or not 1 <= category <= 10 or box_width <= 0 or box_height <= 0:
                continue
            x1, y1 = max(0.0, x), max(0.0, y)
            x2, y2 = min(float(width), x + box_width), min(float(height), y + box_height)
            if x2 <= x1 or y2 <= y1:
                continue
            clipped_width, clipped_height = x2 - x1, y2 - y1
            annotations.append({
                'id': annotation_id, 'image_id': image_id, 'category_id': category - 1,
                'bbox': [x1, y1, clipped_width, clipped_height],
                'area': clipped_width * clipped_height, 'iscrowd': 0, 'segmentation': []
            })
            annotation_id += 1
    payload = {
        'info': {'description': 'VisDrone converted for COCO bbox evaluation'},
        'images': images,
        'annotations': annotations,
        'categories': [{'id': index, 'name': name} for index, name in enumerate(CLASS_NAMES)]
    }
    output_path.write_text(json.dumps(payload), encoding='utf-8')
    return output_path

COCO_JSONS = {
    split: convert_visdrone_to_coco(root, COCO_DIR / f'{split}.json')
    for split, root in RAW_SPLITS.items()
}
print(COCO_JSONS)

{'train': '/kaggle/input/datasets/kushagrapandya/visdrone-dataset/VisDrone2019-DET-train/VisDrone2019-DET-train', 'val': '/kaggle/input/datasets/kushagrapandya/visdrone-dataset/VisDrone2019-DET-val/VisDrone2019-DET-val', 'test': '/kaggle/input/datasets/kushagrapandya/visdrone-dataset/VisDrone2019-DET-test-dev/VisDrone2019-DET-test-dev'}


COCO VisDrone2019-DET-train:   0%|          | 0/6471 [00:00<?, ?it/s]

COCO VisDrone2019-DET-val:   0%|          | 0/548 [00:00<?, ?it/s]

COCO VisDrone2019-DET-test-dev:   0%|          | 0/1610 [00:00<?, ?it/s]

{'train': PosixPath('/kaggle/working/rtdetr_r50vd_o365_visdrone/coco/train.json'), 'val': PosixPath('/kaggle/working/rtdetr_r50vd_o365_visdrone/coco/val.json'), 'test': PosixPath('/kaggle/working/rtdetr_r50vd_o365_visdrone/coco/test.json')}


In [3]:
# 3) Letterbox and train-only on-the-fly slicing (probability 0.7).
@dataclass
class LetterboxMeta:
    image_id: int
    file_name: str
    orig_width: int
    orig_height: int
    scale: float
    pad_left: int
    pad_top: int

def clip_boxes(boxes, width, height, min_size=TRAIN_SLICE_MIN_BOX_SIZE):
    boxes = np.asarray(boxes, dtype=np.float32).reshape(-1, 4).copy()
    if not len(boxes):
        return boxes, np.empty(0, dtype=bool)
    x1 = np.clip(boxes[:, 0], 0, width)
    y1 = np.clip(boxes[:, 1], 0, height)
    x2 = np.clip(boxes[:, 0] + boxes[:, 2], 0, width)
    y2 = np.clip(boxes[:, 1] + boxes[:, 3], 0, height)
    clipped = np.stack([x1, y1, np.maximum(0, x2 - x1), np.maximum(0, y2 - y1)], axis=1)
    keep = (clipped[:, 2] >= min_size) & (clipped[:, 3] >= min_size)
    return clipped[keep].astype(np.float32), keep

def letterbox(image, boxes, size=IMAGE_SIZE):
    source_width, source_height = image.size
    scale = min(size / source_width, size / source_height)
    resized_width = max(1, int(round(source_width * scale)))
    resized_height = max(1, int(round(source_height * scale)))
    left, top = (size - resized_width) // 2, (size - resized_height) // 2
    canvas = Image.new('RGB', (size, size), (PAD_VALUE,) * 3)
    canvas.paste(image.resize((resized_width, resized_height), Image.Resampling.BILINEAR), (left, top))
    boxes = np.asarray(boxes, dtype=np.float32).reshape(-1, 4).copy()
    if len(boxes):
        boxes[:, 0] = (boxes[:, 0] * scale + left + boxes[:, 2] * scale / 2) / size
        boxes[:, 1] = (boxes[:, 1] * scale + top + boxes[:, 3] * scale / 2) / size
        boxes[:, 2:] = boxes[:, 2:] * scale / size
        boxes = np.clip(boxes, 0, 1)
    pixels = torch.from_numpy(np.asarray(canvas, dtype=np.float32).copy()).permute(2, 0, 1) / 255.0
    mask = torch.zeros((size, size), dtype=torch.bool)
    mask[top:top + resized_height, left:left + resized_width] = True
    return pixels, mask, boxes, scale, left, top

def randint_inclusive(low, high):
    low, high = int(low), int(high)
    return low if high <= low else int(torch.randint(low, high + 1, (1,)).item())

def guided_start(box_start, box_length, image_length, crop_length):
    maximum = max(0, int(image_length - crop_length))
    low = max(0, int(math.ceil(box_start + box_length - crop_length)))
    high = min(maximum, int(math.floor(box_start)))
    if low <= high:
        return randint_inclusive(low, high)
    return int(np.clip(round(box_start + box_length / 2 - crop_length / 2), 0, maximum))

def slice_training_sample(image, boxes, classes):
    width, height = image.size
    crop_width, crop_height = min(TRAIN_TILE_SIZE, width), min(TRAIN_TILE_SIZE, height)
    if (crop_width == width and crop_height == height) or torch.rand(1).item() >= TRAIN_SLICE_PROBABILITY:
        return image, boxes, classes, False
    eligible = np.flatnonzero((boxes[:, 2] <= crop_width) & (boxes[:, 3] <= crop_height)) if len(boxes) else np.empty(0, dtype=np.int64)
    if len(eligible) and torch.rand(1).item() < TRAIN_SLICE_OBJECT_GUIDED_PROBABILITY:
        selected = int(eligible[int(torch.randint(len(eligible), (1,)).item())])
        x, y, box_width, box_height = boxes[selected]
        x0 = guided_start(x, box_width, width, crop_width)
        y0 = guided_start(y, box_height, height, crop_height)
    else:
        x0 = randint_inclusive(0, width - crop_width)
        y0 = randint_inclusive(0, height - crop_height)
    if len(boxes):
        x1, y1 = np.maximum(boxes[:, 0], x0), np.maximum(boxes[:, 1], y0)
        x2 = np.minimum(boxes[:, 0] + boxes[:, 2], x0 + crop_width)
        y2 = np.minimum(boxes[:, 1] + boxes[:, 3], y0 + crop_height)
        clipped_width, clipped_height = np.maximum(0, x2 - x1), np.maximum(0, y2 - y1)
        visibility = (clipped_width * clipped_height) / np.maximum(boxes[:, 2] * boxes[:, 3], 1e-6)
        keep = ((visibility >= TRAIN_SLICE_MIN_VISIBILITY) &
                (clipped_width >= TRAIN_SLICE_MIN_BOX_SIZE) &
                (clipped_height >= TRAIN_SLICE_MIN_BOX_SIZE))
        sliced_boxes = np.stack([x1 - x0, y1 - y0, clipped_width, clipped_height], axis=1)[keep].astype(np.float32)
        sliced_classes = classes[keep]
    else:
        sliced_boxes = np.empty((0, 4), dtype=np.float32)
        sliced_classes = classes
    accept_empty = not len(boxes) or torch.rand(1).item() < TRAIN_EMPTY_SLICE_PROBABILITY
    if not len(sliced_boxes) and not accept_empty:
        return image, boxes, classes, False
    return image.crop((x0, y0, x0 + crop_width, y0 + crop_height)), sliced_boxes, sliced_classes, True

class VisDroneDataset(Dataset):
    def __init__(self, image_dir, coco_json, train=False):
        self.image_dir = Path(image_dir)
        data = json.loads(Path(coco_json).read_text(encoding='utf-8'))
        self.images = sorted(data['images'], key=lambda item: item['id'])
        self.by_image = defaultdict(list)
        for annotation in data['annotations']:
            self.by_image[annotation['image_id']].append(annotation)
        self.train = train
    def __len__(self):
        return len(self.images)
    def __getitem__(self, index):
        info = self.images[index]
        image = Image.open(self.image_dir / info['file_name']).convert('RGB')
        annotations = self.by_image[info['id']]
        boxes = np.asarray([item['bbox'] for item in annotations], dtype=np.float32).reshape(-1, 4)
        classes = np.asarray([item['category_id'] for item in annotations], dtype=np.int64)
        if self.train:
            image, boxes, classes, _ = slice_training_sample(image, boxes, classes)
            sample_width, sample_height = image.size
            if torch.rand(1).item() < HORIZONTAL_FLIP_PROBABILITY:
                image = ImageOps.mirror(image)
                if len(boxes):
                    boxes[:, 0] = sample_width - boxes[:, 0] - boxes[:, 2]
            boxes, keep = clip_boxes(boxes, sample_width, sample_height)
            classes = classes[keep]
        else:
            sample_width, sample_height = image.size
        pixels, mask, boxes, scale, left, top = letterbox(image, boxes)
        labels = {
            'class_labels': torch.as_tensor(classes, dtype=torch.long),
            'boxes': torch.as_tensor(boxes, dtype=torch.float32)
        }
        meta = LetterboxMeta(info['id'], info['file_name'], sample_width, sample_height, scale, left, top)
        return pixels, mask, labels, meta

def collate_fn(batch):
    pixels, masks, labels, metas = zip(*batch)
    return {
        'pixel_values': torch.stack(pixels), 'pixel_mask': torch.stack(masks),
        'labels': list(labels), 'metas': list(metas)
    }

train_ds = VisDroneDataset(RAW_SPLITS['train'] / 'images', COCO_JSONS['train'], True)
val_ds = VisDroneDataset(RAW_SPLITS['val'] / 'images', COCO_JSONS['val'], False)
print('train:', len(train_ds), 'val:', len(val_ds), 'slice probability:', TRAIN_SLICE_PROBABILITY)

train: 6471 val: 548 slice probability: 0.7


In [4]:
# 4) Load the complete pretrained RT-DETR and adapt only its class heads to VisDrone.
COCO_ALIASES = {
    'pedestrian': ['person'], 'people': ['person'], 'bicycle': ['bicycle'],
    'car': ['car'], 'van': ['car'], 'truck': ['truck'],
    'tricycle': ['bicycle', 'motorcycle', 'motorbike'],
    'awning-tricycle': ['motorcycle', 'motorbike', 'bicycle'],
    'bus': ['bus'], 'motor': ['motorcycle', 'motorbike']
}

def copy_class_rows(old_layer, new_layer, id2label):
    lookup = {str(value).lower(): int(key) for key, value in id2label.items()}
    with torch.no_grad():
        for index, name in enumerate(CLASS_NAMES):
            source = next((lookup[alias] for alias in COCO_ALIASES[name] if alias in lookup), None)
            if source is not None:
                new_layer.weight[index].copy_(old_layer.weight[source])
                if getattr(old_layer, 'bias', None) is not None:
                    new_layer.bias[index].copy_(old_layer.bias[source])

model = RTDetrForObjectDetection.from_pretrained(
    MODEL_DIR, local_files_only=True, use_safetensors=True
)
source_labels = dict(model.config.id2label)
new_class_heads = nn.ModuleList()
for old_head in model.model.decoder.class_embed:
    new_head = nn.Linear(old_head.in_features, NUM_CLASSES)
    copy_class_rows(old_head, new_head, source_labels)
    new_class_heads.append(new_head)
model.model.decoder.class_embed = new_class_heads

old_score_head = model.model.enc_score_head
new_score_head = nn.Linear(old_score_head.in_features, NUM_CLASSES)
copy_class_rows(old_score_head, new_score_head, source_labels)
model.model.enc_score_head = new_score_head

old_denoising = model.model.denoising_class_embed
new_denoising = nn.Embedding(NUM_CLASSES + 1, old_denoising.embedding_dim, padding_idx=NUM_CLASSES)
copy_class_rows(old_denoising, new_denoising, source_labels)
with torch.no_grad():
    new_denoising.weight[NUM_CLASSES].copy_(old_denoising.weight[-1])
model.model.denoising_class_embed = new_denoising
model.config.num_labels = NUM_CLASSES
model.config.id2label = {index: name for index, name in enumerate(CLASS_NAMES)}
model.config.label2id = {name: index for index, name in enumerate(CLASS_NAMES)}
model.config.architectures = ['RTDetrForObjectDetection']

def make_train_loader():
    return DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
        pin_memory=True, persistent_workers=NUM_WORKERS > 0,
        collate_fn=collate_fn, drop_last=True
    )

def move_labels(labels):
    return [{key: value.to(DEVICE, non_blocking=True) for key, value in item.items()} for item in labels]

train_loader = make_train_loader()
model.to(DEVICE)
smoke_batch = next(iter(train_loader))
model.train()
with torch.no_grad(), torch.autocast(
    device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None
):
    smoke = model(
        pixel_values=smoke_batch['pixel_values'].to(DEVICE),
        pixel_mask=smoke_batch['pixel_mask'].to(DEVICE),
        labels=move_labels(smoke_batch['labels'])
    )
assert smoke.logits.shape[-1] == NUM_CLASSES and torch.isfinite(smoke.loss)
print('parameters:', sum(parameter.numel() for parameter in model.parameters()) / 1e6, 'M')
print('smoke:', smoke.logits.shape, smoke.pred_boxes.shape, float(smoke.loss))
del smoke, smoke_batch
gc.collect()
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()

groups = {'head': [], 'core': [], 'backbone': []}
for name, parameter in model.named_parameters():
    if name.startswith('model.backbone.'):
        groups['backbone'].append(parameter)
    elif 'class_embed' in name or 'enc_score_head' in name or 'denoising_class_embed' in name:
        groups['head'].append(parameter)
    else:
        groups['core'].append(parameter)
optimizer = torch.optim.AdamW([
    {'params': groups['head'], 'lr': LR_HEAD},
    {'params': groups['core'], 'lr': LR_CORE},
    {'params': groups['backbone'], 'lr': LR_BACKBONE}
], weight_decay=WEIGHT_DECAY)
updates_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=max(1, EPOCHS * updates_per_epoch), eta_min=LR_BACKBONE * 0.05
)
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == 'cuda' and AMP_DTYPE == torch.float16)

Loading weights:   0%|          | 0/764 [00:00<?, ?it/s]

parameters: 42.747522 M
smoke: torch.Size([48, 784, 10]) torch.Size([48, 784, 4]) 25.96796226501465


/tmp/ipykernel_65/858171208.py:94: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == 'cuda' and AMP_DTYPE == torch.float16)


In [5]:
# 5) GOIS tiled inference and COCO bbox evaluation only.
def decode_one(boxes, logits, meta):
    scores, labels = logits.sigmoid().max(-1)
    center_x, center_y, width, height = boxes.unbind(-1)
    x1 = ((center_x - width / 2) * IMAGE_SIZE - meta.pad_left) / meta.scale
    y1 = ((center_y - height / 2) * IMAGE_SIZE - meta.pad_top) / meta.scale
    x2 = ((center_x + width / 2) * IMAGE_SIZE - meta.pad_left) / meta.scale
    y2 = ((center_y + height / 2) * IMAGE_SIZE - meta.pad_top) / meta.scale
    x1, x2 = x1.clamp(0, meta.orig_width), x2.clamp(0, meta.orig_width)
    y1, y2 = y1.clamp(0, meta.orig_height), y2.clamp(0, meta.orig_height)
    width, height = x2 - x1, y2 - y1
    keep = (width > 0.5) & (height > 0.5)
    rows = torch.stack([
        x1, y1, width, height, scores, labels.float()
    ], -1)[keep]
    if len(rows):
        rows = rows[torch.argsort(rows[:, 4], descending=True)]
    return rows.cpu().numpy()

def tile_starts(length):
    if length <= GOIS_TILE_SIZE:
        return [0]
    stride = max(1, int(round(GOIS_TILE_SIZE * (1 - GOIS_OVERLAP))))
    starts = list(range(0, length - GOIS_TILE_SIZE + 1, stride))
    final = length - GOIS_TILE_SIZE
    if starts[-1] != final:
        starts.append(final)
    return starts

def merge_tile_rows(chunks):
    if not chunks:
        return np.empty((0, 6), dtype=np.float32)
    rows = torch.as_tensor(np.concatenate(chunks), dtype=torch.float32)
    rows = rows[rows[:, 4] >= GOIS_SCORE_THRESHOLD]
    if not len(rows):
        return np.empty((0, 6), dtype=np.float32)
    boxes = torch.stack([rows[:, 0], rows[:, 1], rows[:, 0] + rows[:, 2], rows[:, 1] + rows[:, 3]], -1)
    keep = batched_nms(boxes, rows[:, 4], rows[:, 5], GOIS_NMS_IOU)[:GOIS_MAX_DETECTIONS]
    rows = rows[keep]
    return rows[torch.argsort(rows[:, 4], descending=True)].numpy()

@torch.inference_mode()
def write_predictions_gois(model, dataset, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    model.eval()
    predictions = []
    started, tile_count = time.perf_counter(), 0
    for info in tqdm(dataset.images, desc='GOIS inference', unit='image'):
        image = Image.open(dataset.image_dir / info['file_name']).convert('RGB')
        specs = [
            (x, y, min(x + GOIS_TILE_SIZE, info['width']), min(y + GOIS_TILE_SIZE, info['height']))
            for y in tile_starts(info['height']) for x in tile_starts(info['width'])
        ]
        chunks = []
        for start in range(0, len(specs), GOIS_TILE_BATCH_SIZE):
            current = specs[start:start + GOIS_TILE_BATCH_SIZE]
            pixels, masks, metas = [], [], []
            for x1, y1, x2, y2 in current:
                tile = image.crop((x1, y1, x2, y2))
                pixel, mask, _, scale, left, top = letterbox(tile, np.empty((0, 4), dtype=np.float32))
                pixels.append(pixel)
                masks.append(mask)
                metas.append(LetterboxMeta(info['id'], info['file_name'], tile.width, tile.height, scale, left, top))
            with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
                output = model(
                    pixel_values=torch.stack(pixels).to(DEVICE),
                    pixel_mask=torch.stack(masks).to(DEVICE)
                )
            for spec_box, boxes, logits, meta in zip(current, output.pred_boxes.float(), output.logits.float(), metas):
                rows = decode_one(boxes, logits, meta)
                if len(rows):
                    rows[:, 0] += spec_box[0]
                    rows[:, 1] += spec_box[1]
                    chunks.append(rows)
            tile_count += len(current)
        merged = merge_tile_rows(chunks)
        for x, y, width, height, score, category in merged:
            predictions.append({
                'image_id': int(info['id']), 'category_id': int(category),
                'bbox': [float(x), float(y), float(width), float(height)],
                'score': float(score)
            })
    elapsed = time.perf_counter() - started
    if not predictions:
        raise RuntimeError(f'No detections were produced for {output_path.name}.')
    output_path.write_text(json.dumps(predictions), encoding='utf-8')
    return output_path, {
        'seconds': elapsed, 'images': len(dataset), 'tiles': tile_count,
        'detections': len(predictions), 'seconds_per_image': elapsed / max(1, len(dataset))
    }

COCO_METRIC_NAMES = [
    'AP', 'AP50', 'AP75', 'AP_small', 'AP_medium', 'AP_large',
    'AR_1', 'AR_10', 'AR_100', 'AR_small', 'AR_medium', 'AR_large'
]

def run_coco_eval(ground_truth_path, prediction_path):
    coco_gt = COCO(str(ground_truth_path))
    coco_dt = coco_gt.loadRes(str(prediction_path))
    evaluator = COCOeval(coco_gt, coco_dt, iouType='bbox')
    evaluator.params.imgIds = sorted(coco_gt.getImgIds())
    evaluator.params.catIds = sorted(coco_gt.getCatIds())
    evaluator.params.maxDets = [1, 10, 100]
    evaluator.evaluate()
    evaluator.accumulate()
    evaluator.summarize()
    metrics = {name: float(value) for name, value in zip(COCO_METRIC_NAMES, evaluator.stats)}
    return {
        'evaluator': 'pycocotools.COCOeval', 'iou_type': 'bbox',
        'unit': 'raw_0_1', 'metrics': metrics,
        'metrics_percent': {name: value * 100.0 for name, value in metrics.items()},
        'max_detections': list(evaluator.params.maxDets)
    }

def evaluate_split_gois(model, split, tag):
    dataset = val_ds if split == 'val' else VisDroneDataset(
        RAW_SPLITS[split] / 'images', COCO_JSONS[split], False
    )
    prediction_path, timing = write_predictions_gois(
        model, dataset, PRED_DIR / f'{split}_{tag}.json'
    )
    coco = run_coco_eval(COCO_JSONS[split], prediction_path)
    result = {
        'inference': 'GOIS tiled inference', 'timing': timing,
        'predictions': str(prediction_path), 'coco': coco
    }
    (WORK_DIR / f'metrics_{split}_{tag}.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
    print(split, 'COCO %:', coco['metrics_percent'])
    return result

In [6]:
# 6) Fine-tune and select best.pt using validation COCO AP.
ARCHITECTURE = {
    'name': 'RT-DETR-R50VD',
    'pretrained_model': EXPECTED_MODEL_ID,
    'pretraining': 'Objects365 + COCO',
    'image_size': IMAGE_SIZE,
    'classes': CLASS_NAMES
}
SLICING_CONFIG = {
    'mode': 'on_the_fly_train_only', 'tile_size': TRAIN_TILE_SIZE,
    'probability': TRAIN_SLICE_PROBABILITY,
    'object_guided_probability': TRAIN_SLICE_OBJECT_GUIDED_PROBABILITY,
    'min_visibility': TRAIN_SLICE_MIN_VISIBILITY
}

def set_backbone_trainable(enabled):
    for parameter in model.model.backbone.parameters():
        parameter.requires_grad_(enabled)

def save_checkpoint(path, epoch, best_validation_coco_ap, history):
    torch.save({
        'architecture': ARCHITECTURE,
        'training_slicing': SLICING_CONFIG,
        'selection_metric': 'validation COCO AP@[0.50:0.95]',
        'epoch': epoch,
        'best_validation_coco_ap': best_validation_coco_ap,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'scaler': scaler.state_dict(),
        'history': history,
        'versions': {'torch': torch.__version__, 'transformers': transformers.__version__}
    }, path)

start_epoch, best_validation_coco_ap, history = 0, -float('inf'), []
if RESUME_CHECKPOINT:
    checkpoint = torch.load(RESUME_CHECKPOINT, map_location='cpu', weights_only=False)
    model.load_state_dict(checkpoint['model'])
    optimizer.load_state_dict(checkpoint['optimizer'])
    scheduler.load_state_dict(checkpoint['scheduler'])
    scaler.load_state_dict(checkpoint['scaler'])
    start_epoch = checkpoint['epoch'] + 1
    best_validation_coco_ap = checkpoint.get('best_validation_coco_ap', -float('inf'))
    history = checkpoint.get('history', [])

for epoch in range(start_epoch, EPOCHS):
    freeze_backbone = epoch < BACKBONE_FROZEN_EPOCHS
    set_backbone_trainable(not freeze_backbone)
    model.train()
    if freeze_backbone:
        model.model.backbone.eval()
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0
    progress = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{EPOCHS}')
    for step, batch in enumerate(progress):
        pixels = batch['pixel_values'].to(DEVICE, non_blocking=True)
        masks = batch['pixel_mask'].to(DEVICE, non_blocking=True)
        labels = move_labels(batch['labels'])
        with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=AMP_DTYPE is not None):
            output = model(pixel_values=pixels, pixel_mask=masks, labels=labels)
            loss = output.loss / GRAD_ACCUM_STEPS
        scaler.scale(loss).backward()
        running_loss += float(loss.detach()) * GRAD_ACCUM_STEPS
        update = (step + 1) % GRAD_ACCUM_STEPS == 0 or step + 1 == len(train_loader)
        if update:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
        progress.set_postfix(loss=f'{running_loss / (step + 1):.4f}')

    row = {'epoch': epoch + 1, 'train_loss': running_loss / len(train_loader), 'validation_coco_AP': None}
    if (epoch + 1) % EVAL_EVERY == 0 or epoch + 1 == EPOCHS:
        validation = evaluate_split_gois(model, 'val', f'epoch_{epoch + 1:03d}')
        validation_coco_ap = validation['coco']['metrics']['AP']
        row['validation_coco_AP'] = validation_coco_ap
        if validation_coco_ap > best_validation_coco_ap:
            best_validation_coco_ap = validation_coco_ap
            save_checkpoint(CHECKPOINT_DIR / 'best.pt', epoch, best_validation_coco_ap, history + [row])
            print('New best validation COCO AP:', best_validation_coco_ap)
    history.append(row)
    save_checkpoint(CHECKPOINT_DIR / 'last.pt', epoch, best_validation_coco_ap, history)
    (WORK_DIR / 'history.json').write_text(json.dumps(history, indent=2), encoding='utf-8')

Epoch 1/20:   0%|          | 0/134 [00:00<?, ?it/s]

GOIS inference:   0%|          | 0/548 [00:00<?, ?image/s]

loading annotations into memory...
Done (t=0.06s)
creating index...
index created!
Loading and preparing results...
DONE (t=1.18s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=14.51s).
Accumulating evaluation results...
DONE (t=0.61s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.148
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.250
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.148
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.097
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.210
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.350
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.057
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.171
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDet

Epoch 2/20:   0%|          | 0/134 [00:00<?, ?it/s]

GOIS inference:   0%|          | 0/548 [00:00<?, ?image/s]

...

GOIS inference:   0%|          | 0/548 [00:00<?, ?image/s]

loading annotations into memory...
Done (t=0.06s)
creating index...
index created!
Loading and preparing results...
DONE (t=1.36s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=15.65s).
Accumulating evaluation results...
DONE (t=0.65s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.239
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.430
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.230
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.168
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.343
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.427
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.104
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.305
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDet

Epoch 20/20:   0%|          | 0/134 [00:00<?, ?it/s]

GOIS inference:   0%|          | 0/548 [00:00<?, ?image/s]

loading annotations into memory...
Done (t=0.07s)
creating index...
index created!
Loading and preparing results...
DONE (t=1.33s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=15.39s).
Accumulating evaluation results...
DONE (t=0.65s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.239
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.429
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.231
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.168
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.343
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.432
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.104
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.304
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDet

In [7]:
# 7) Load best.pt and evaluate test-dev only (do not rerun validation).
BEST_CHECKPOINT = CHECKPOINT_DIR / 'best.pt'
if not BEST_CHECKPOINT.is_file():
    raise FileNotFoundError('Training did not produce best.pt.')
selected = torch.load(BEST_CHECKPOINT, map_location='cpu', weights_only=False)
model.load_state_dict(selected['model'])
model.to(DEVICE)
print('Selected best epoch:', selected['epoch'] + 1)
print('Selected validation COCO AP:', selected['best_validation_coco_ap'])

test_metrics = evaluate_split_gois(model, 'test', 'best_test_dev')
summary = {
    'architecture': ARCHITECTURE,
    'training_slicing': SLICING_CONFIG,
    'best_checkpoint': str(BEST_CHECKPOINT),
    'best_epoch': selected['epoch'] + 1,
    'best_validation_coco_AP': selected['best_validation_coco_ap'],
    'test_dev': test_metrics,
    'history': selected.get('history', history)
}
SUMMARY_PATH = WORK_DIR / 'run_summary.json'
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding='utf-8')
archive = shutil.make_archive(
    '/kaggle/working/rtdetr_r50vd_o365_visdrone_checkpoints', 'zip', CHECKPOINT_DIR
)
print('Test-dev COCO %:', test_metrics['coco']['metrics_percent'])
print('Summary:', SUMMARY_PATH)
print('Checkpoint archive:', archive)

Selected best epoch: 9
Selected validation COCO AP: 0.24582923030007553


GOIS inference:   0%|          | 0/1610 [00:00<?, ?image/s]

loading annotations into memory...
Done (t=0.12s)
creating index...
index created!
Loading and preparing results...
DONE (t=3.69s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=32.03s).
Accumulating evaluation results...
DONE (t=1.91s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.195
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.351
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.188
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.111
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.300
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.386
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.084
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.269
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDet